# High-accuracy dash-cam pothole model — RDD2022 YOLOv8s (Ayaan's plan)

Why: on real Detroit dash-cam clips the current model boxes **manhole covers and house siding** and
misses real damage. This run fixes that with:

1. **RDD2022 YOLOv8s starting weights** (`dronefreak/rdd2022-yolov8s`, AGPL-3.0 — credit in `pipeline/README.md`):
   already trained on tens of thousands of car-mounted road images.
2. **More pothole data:** Roboflow `voxrl`, `dashcam`, `hitesh`, `vit` + **Figshare 2000+ dash-cam frames**.
   Everything becomes one class `pothole` (fits Ado's `?m=dashcam` slot, no code change).
3. **"Nothing here" frames** (hard negatives) from Detroit footage — empty labels → stops manhole/siding boxes.
4. Ayaan's training settings (640, 60 epochs, gentle lr, grey-light colour jitter).
5. Old vs new on the **same held-out dash-cam frames**, a **tracked demo video**, and the ONNX export.

**Setup:** A100 (or L4) runtime · 🔑 `ROBOFLOW_API_KEY` in Secrets · Detroit clips (.mp4) in
`My Drive/still-there-potholes/test-videos/` (Ayaan's CROWD clips or yours) · `Runtime → Run all`.

**Two-pass flow for the negatives (step 3):** the first Run all collects *candidate* frames where the current
model fires on Detroit footage and **stops**. Open `still-there-potholes/negatives/` in Drive, **delete every
frame that really shows a pothole** (keep manholes, siding, patches, shadows), then Run all again — it skips
straight to training. ~45–75 min on an A100 (~10–15 units).

## 1 · GPU + Drive + install

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU"
!pip -q install ultralytics roboflow huggingface_hub
from google.colab import drive
drive.mount('/content/drive')
import os, glob
DRIVE_DIR = '/content/drive/MyDrive/still-there-potholes'
CUR_BEST = next((p for p in (f'{DRIVE_DIR}/handoff_dashcam/best.pt', f'{DRIVE_DIR}/handoff/best.pt') if os.path.exists(p)), None)
print('current model (baseline + negative miner):', CUR_BEST)
VIDEOS = sorted(glob.glob(f'{DRIVE_DIR}/test-videos/*.mp4') + glob.glob(f'{DRIVE_DIR}/test-videos/*.MP4'))
print(len(VIDEOS), 'Detroit clip(s) in test-videos/')

## 2 · Download: RDD2022 weights, 4 Roboflow sets, Figshare dash-cam

In [ ]:
import zipfile, requests, getpass
from huggingface_hub import hf_hub_download, list_repo_files
RAW = '/content/datasets'; os.makedirs(RAW, exist_ok=True)

# RDD2022 YOLOv8s weights
files = list_repo_files('dronefreak/rdd2022-yolov8s')
pt = next((f for f in files if f.endswith('best.pt')), next(f for f in files if f.endswith('.pt')))
RDD_WEIGHTS = hf_hub_download('dronefreak/rdd2022-yolov8s', pt)
print('RDD2022 weights:', RDD_WEIGHTS)

# Roboflow (your own key)
try:
    from google.colab import userdata; RF_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    RF_KEY = None
if isinstance(RF_KEY, dict): RF_KEY = RF_KEY.get('value') or next((v for v in RF_KEY.values() if isinstance(v, str)), None)
RF_KEY = RF_KEY if isinstance(RF_KEY, str) and RF_KEY else getpass.getpass('Roboflow API key: ')
from roboflow import Roboflow
rf = Roboflow(api_key=RF_KEY)
RF_SETS = {
    'voxrl':   ('brad-dwyer', 'pothole-voxrl', 1),
    'dashcam': ('work-5qhak', 'potholes-ptvvw', 2),
    'hitesh':  ('hiteshram', 'object-detection-bounding-box-ftfs5', 1),
    'vit':     ('vit-76kid', 'pothole-detection-project-3yiqt', 1),
}
RF_PATHS = {}
for name, (ws, proj, ver) in RF_SETS.items():
    loc = f'{RAW}/{name}'
    if not os.path.exists(f'{loc}/data.yaml'):
        try:
            v = rf.workspace(ws).project(proj).version(ver)
            try: v.download('yolov11', location=loc)
            except Exception as e: print(name, 'yolov11 failed, using yolov8:', e); v.download('yolov8', location=loc)
        except Exception as e:
            print(f'!! {name} could not be downloaded, skipping it: {e}'); continue
    RF_PATHS[name] = loc
print('roboflow sets:', list(RF_PATHS))

# Figshare dash-cam set
FS = f'{RAW}/figshare_dashcam'; os.makedirs(FS, exist_ok=True)
meta = requests.get('https://api.figshare.com/v2/articles/21214400', timeout=30).json()
for f in meta.get('files', []):
    dst = f"{FS}/{f['name']}"
    if os.path.exists(dst) and os.path.getsize(dst) == f['size']: continue
    print(f"downloading {f['name']} ({f['size']/1e6:.0f} MB)...")
    with requests.get(f['download_url'], stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(dst, 'wb') as out:
            for chunk in r.iter_content(1 << 20): out.write(chunk)
seen = set()
while True:
    zips = [z for z in glob.glob(f'{FS}/**/*.zip', recursive=True) if z not in seen]
    if not zips: break
    for z in zips: seen.add(z); zipfile.ZipFile(z).extractall(os.path.splitext(z)[0])
print('downloads done')

## 3 · "Nothing here" frames from Detroit footage (hard negatives)

Pass 1: grabs 1 frame/s from each clip in `test-videos/`, keeps frames where the **current** model draws a
box (conf ≥ 0.25), saves them to `negatives/` (+ a boxed preview in `negatives/_preview/` so you can see
what it fired on), then **stops**. Delete the real-pothole frames from `negatives/` (preview names match),
then Run all again. Pass 2: whatever is left in `negatives/` is used as empty-label training images.
To skip negatives entirely, create an empty file `negatives/DONE`.

In [ ]:
import cv2
from ultralytics import YOLO
NEG = f'{DRIVE_DIR}/negatives'; os.makedirs(f'{NEG}/_preview', exist_ok=True)
DONE_FLAG = f'{NEG}/DONE'
if not os.path.exists(DONE_FLAG):
    if not VIDEOS:
        print('No clips in test-videos/ — skipping negatives. (Add Detroit clips and re-run for best results.)')
        open(DONE_FLAG, 'w').close()
    else:
        miner = YOLO(CUR_BEST); n = 0
        for v in VIDEOS:
            cap = cv2.VideoCapture(v); fps = cap.get(cv2.CAP_PROP_FPS) or 30; i = 0
            while True:
                ok, frame = cap.read()
                if not ok: break
                if i % int(round(fps)) == 0:
                    r = miner.predict(frame, conf=0.25, imgsz=640, verbose=False)[0]
                    if len(r.boxes):
                        name = f'{os.path.splitext(os.path.basename(v))[0]}_{i:06d}.jpg'
                        cv2.imwrite(f'{NEG}/{name}', frame); cv2.imwrite(f'{NEG}/_preview/{name}', r.plot()); n += 1
                i += 1
            cap.release()
        open(DONE_FLAG, 'w').close()
        raise SystemExit(f'Saved {n} candidate frames to {NEG}. Open Drive → negatives/_preview, delete the matching '
                         f'frame in negatives/ for every REAL pothole, then Runtime → Run all again.')
NEG_IMGS = [p for p in glob.glob(f'{NEG}/*.jpg')]
print(len(NEG_IMGS), '"nothing here" frames will be added to training')

## 4 · Build the dataset (one class `pothole`, Figshare split by video stretch, negatives → train only)

In [ ]:
BUILD_RDD = True  # marker for the offline test
import shutil, random, yaml
from collections import Counter
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp')
OUT = '/content/rdd_data'

def pothole_ids(names):
    names = list(names.values()) if isinstance(names, dict) else list(names)
    other = ('crack', 'manhole', 'patch', 'bump', 'sign', 'car', 'person', 'drain')
    if len(names) == 1 and not any(w in str(names[0]).lower() for w in other): return {0}, names
    return {i for i, n in enumerate(names) if 'pothole' in str(n).lower()}, names

def rows_from(label_path, keep=None):
    rows = []
    if not label_path or not os.path.exists(label_path): return rows
    for line in open(label_path):
        p = line.split()
        if len(p) < 5: continue
        if keep is not None and int(float(p[0])) not in keep: continue
        nums = [float(x) for x in p[1:]]
        if len(nums) > 4:
            xs, ys = nums[0::2], nums[1::2]
            nums = [(min(xs)+max(xs))/2, (min(ys)+max(ys))/2, max(xs)-min(xs), max(ys)-min(ys)]
        rows.append('0 ' + ' '.join(f'{v:.6f}' for v in nums[:4]))
    return rows

def put(split, img, rows, name, stats):
    for d in ('images', 'labels'): os.makedirs(f'{OUT}/{split}/{d}', exist_ok=True)
    shutil.copy(img, f'{OUT}/{split}/images/{name}{os.path.splitext(img)[1].lower()}')
    with open(f'{OUT}/{split}/labels/{name}.txt', 'w') as f: f.write('\n'.join(rows) + ('\n' if rows else ''))
    stats[split] += 1; stats[f'{split}_boxes'] += len(rows); stats[f'{split}_empty'] += (not rows)

def build_rdd(rf_paths, fs_src, neg_imgs):
    shutil.rmtree(OUT, ignore_errors=True); stats = Counter()
    for tag, root in rf_paths.items():
        keep, names = pothole_ids(yaml.safe_load(open(f'{root}/data.yaml'))['names'])
        print(f'{tag}: classes {names} -> keep {[names[i] for i in keep]}')
        if not keep: print(f'  !! {tag}: no pothole class, skipped'); continue
        for split in ('train', 'valid', 'test'):
            d = next((f'{root}/{c}' for c in (split, 'val') if os.path.isdir(f'{root}/{c}/images')), None)
            if not d: continue
            for img in glob.glob(f'{d}/images/*'):
                stem = os.path.splitext(os.path.basename(img))[0]
                put(split, img, rows_from(f'{d}/labels/{stem}.txt', keep), f'{tag}_{stem}', stats)
    labels = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(f'{fs_src}/**/*.txt', recursive=True)
              if 'classes' not in os.path.basename(p).lower()}
    imgs = sorted(p for p in glob.glob(f'{fs_src}/**/*', recursive=True)
                  if p.lower().endswith(IMG_EXT) and os.path.splitext(os.path.basename(p))[0] in labels)
    a, b = int(len(imgs) * .8), int(len(imgs) * .9)
    for i, img in enumerate(imgs):
        split = 'train' if i < a else 'valid' if i < b else 'test'
        put(split, img, rows_from(labels[os.path.splitext(os.path.basename(img))[0]]), f'fs_{i:05d}', stats)
    fstest = f'{OUT}_fstest'; shutil.rmtree(fstest, ignore_errors=True)
    os.makedirs(f'{fstest}/test/images', exist_ok=True); os.makedirs(f'{fstest}/test/labels', exist_ok=True)
    for p in glob.glob(f'{OUT}/test/images/fs_*'):
        stem = os.path.splitext(os.path.basename(p))[0]
        shutil.copy(p, f'{fstest}/test/images/'); shutil.copy(f'{OUT}/test/labels/{stem}.txt', f'{fstest}/test/labels/')
    for img in neg_imgs:
        put('train', img, [], 'neg_' + os.path.splitext(os.path.basename(img))[0], stats)
    for root, tr, va in ((OUT, 'train/images', 'valid/images'), (fstest, 'test/images', 'test/images')):
        yaml.safe_dump({'path': root, 'train': tr, 'val': va, 'test': 'test/images', 'names': {0: 'pothole'}},
                       open(f'{root}/data.yaml', 'w'))
    print(dict(stats))
    return f'{OUT}/data.yaml', f'{fstest}/data.yaml'

if 'RF_PATHS' in globals():
    DATA_YAML, FSTEST_YAML = build_rdd(RF_PATHS, FS, NEG_IMGS)

**Spot-check the new sets (Ayaan's rule):** run the next cell, look at the grids, and if a set has many
visible potholes *without* a box, remove it from `RF_SETS` in step 2 and re-run steps 2 + 4.

In [ ]:
import matplotlib.pyplot as plt
for tag in [t for t in ('hitesh', 'vit') if t in RF_PATHS]:
    imgs = random.sample(glob.glob(f'{OUT}/train/images/{tag}_*'), min(8, len(glob.glob(f'{OUT}/train/images/{tag}_*'))))
    fig, axes = plt.subplots(2, 4, figsize=(20, 9)); fig.suptitle(tag, fontsize=16)
    for ax, p in zip(axes.flat, imgs):
        im = cv2.imread(p)[..., ::-1]; h, w = im.shape[:2]; ax.imshow(im); ax.axis('off')
        for line in open(p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'):
            _, x, y, bw, bh = map(float, line.split())
            ax.add_patch(plt.Rectangle(((x-bw/2)*w, (y-bh/2)*h), bw*w, bh*h, fill=False, color='lime', lw=2))
    plt.show()

## 5 · Baseline: current model on the held-out dash-cam frames

In [ ]:
def score(weights, imgsz=640):
    m = YOLO(weights).val(data=FSTEST_YAML, split='test', imgsz=imgsz, verbose=False, plots=False)
    return m.box.map50, m.box.mp, m.box.mr
RESULTS = {'current model': score(CUR_BEST)}
print('current model  mAP50={:.3f}  P={:.3f}  R={:.3f}'.format(*RESULTS['current model']))

## 6 · Train from RDD2022 YOLOv8s (Ayaan's settings) — resumes after a disconnect

In [ ]:
RUN = f'{DRIVE_DIR}/runs/dashcam_rdd'
last, done = f'{RUN}/weights/last.pt', f'{RUN}/weights/TRAINING_DONE'
if os.path.exists(done):
    print('Already trained — skipping. Delete', done, 'to train again.')
elif os.path.exists(last):
    YOLO(last).train(resume=True)
else:
    YOLO(RDD_WEIGHTS).train(
        data=DATA_YAML, imgsz=640, epochs=60, patience=15,
        lr0=0.002, warmup_epochs=1,             # fine-tuning: gentler learning rate
        batch=-1, cache='ram', workers=min(8, os.cpu_count() or 2), close_mosaic=10,
        hsv_v=0.5, hsv_s=0.7,                   # grey, wet November light
        project=f'{DRIVE_DIR}/runs', name='dashcam_rdd', exist_ok=True)
open(done, 'w').close()
NEW_BEST = f'{RUN}/weights/best.pt'

## 7 · Old vs new on the same held-out frames

In [ ]:
RESULTS['RDD2022 v8s (new)'] = score(NEW_BEST)
print(f"{'model':22s} {'mAP50':>7s} {'P':>7s} {'R (found)':>10s}")
for k, (m, p, r) in RESULTS.items(): print(f'{k:22s} {m:7.3f} {p:7.3f} {r:10.3f}')

## 8 · Demo video: tracked potholes (stable IDs + count) on your Detroit clips

In [ ]:
for v in VIDEOS:
    ids = set()
    for r in YOLO(NEW_BEST).track(v, conf=0.35, imgsz=640, tracker='bytetrack.yaml', persist=True, stream=True,
                                  save=True, verbose=False, project=f'{DRIVE_DIR}/demo-videos',
                                  name=os.path.splitext(os.path.basename(v))[0], exist_ok=True):
        if r.boxes.id is not None: ids.update(int(i) for i in r.boxes.id)
    print(f'{os.path.basename(v)}: {len(ids)} distinct potholes tracked')
print('annotated videos in', f'{DRIVE_DIR}/demo-videos')

## 9 · Export for Ado's `?m=dashcam` slot

Ship only if step 7 (and Ayaan's `/test` page on a Detroit clip) shows more potholes found **and** fewer
false boxes. Upload: `python3 apps/live/tools/upload_model.py pothole-dashcam.onnx --name pothole-dashcam.onnx`.

In [ ]:
out = f'{DRIVE_DIR}/handoff_rdd'; os.makedirs(out, exist_ok=True)
shutil.copy(YOLO(NEW_BEST).export(format='onnx', imgsz=640), f'{out}/pothole-dashcam.onnx')
shutil.copy(NEW_BEST, f'{out}/best.pt')
for f in sorted(os.listdir(out)): print(f'{f:22s} {os.path.getsize(f"{out}/{f}")/1e6:.1f} MB')
print('\nDONE — now Runtime → Disconnect and delete runtime to stop burning units.')